In [9]:
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)

# Load dataset
df = pd.read_csv('src/Dataset/RawData.csv')

# @dataset{multimodal_sports_injury_202},
#   title={Multimodal Sports Injury Prediction Dataset},
#   author={Anjali Bhegam et al.},
#   year={2025},
#   publisher={Kaggle},
#   url={https://www.kaggle.com/datasets/anjalibhegam/multimodal-sports-injury-dataset}

# Basic info
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
display(df.head())

Rows: 15420 | Columns: 29


,athlete_id,heart_rate,body_temperature,hydration_level,sleep_quality,recovery_score,stress_level,muscle_activity,joint_angles,gait_speed,...,training_intensity,training_duration,training_load,fatigue_index,injury_occurred,session_id,sport_type,gender,age,bmi
0,1,NaN,37.048668,76.481477,5.466370,77.899691,0.375825,10.000000,126.480118,2.654327,...,5.678475,79.106564,548.417962,45.028687,0,1,Basketball,Female,18,25.105748
1,1,40.000000,36.929536,72.363938,6.899090,74.845323,0.279165,310.293156,157.631630,3.431107,...,NaN,116.954654,538.043815,42.254717,2,2,Basketball,Female,18,25.105748
2,1,76.174609,36.334841,100.000000,9.574152,83.538316,0.334061,158.659094,81.766287,0.800000,...,6.643242,96.354742,647.784339,61.870056,0,3,Basketball,Female,18,25.105748
3,1,100.543724,37.244474,86.172473,7.614297,79.966618,0.218766,107.079298,113.023701,1.713869,...,6.194427,97.619130,336.553431,50.268845,0,4,Basketball,Female,18,25.105748
4,1,78.001235,36.510766,83.195889,5.631979,56.945979,0.484086,164.347133,112.814695,2.154233,...,5.835804,83.155527,496.076352,29.233575,2,5,Basketball,Female,18,25.105748


In [11]:
display(df.dtypes.rename("stored_type").to_frame())

missing_by_column = df.isna().sum().sort_values(ascending=False)

print("Missing cells by column:")
display(missing_by_column[missing_by_column > 0].to_frame("missing_count"))

print(f"Total missing cells: {df.isna().sum().sum()}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")

,stored_type
athlete_id,int64
heart_rate,float64
body_temperature,float64
hydration_level,float64
sleep_quality,float64
recovery_score,float64
stress_level,float64
muscle_activity,float64
joint_angles,float64
gait_speed,float64


Missing cells by column:


,missing_count
training_intensity,632
muscle_activity,570
sleep_quality,493
gait_speed,431
hydration_level,385
heart_rate,354


Total missing cells: 2865
Exact duplicate rows: 0


In [34]:
summary = df.describe().T
iqr = summary['75%'] - summary['25%']

# Calculate bounds
lower = summary['25%'] - 1.5 * iqr
upper = summary['75%'] + 1.5 * iqr

# Filter numeric columns to match summary's index
num_df = df[summary.index]

# Add the new column
summary['outlier_pct'] = ((num_df < lower) | (num_df > upper)).mean() * 100

display(summary)

,count,mean,std,min,25%,50%,75%,max,outlier_pct
athlete_id,15420.0,78.547601,45.070131,1.000000,39.000000,79.000000,118.000000,156.000000,0.000000
heart_rate,15066.0,72.688277,17.758991,40.000000,60.097967,72.304051,84.812895,159.270125,0.285344
body_temperature,15420.0,37.106443,0.594843,35.800000,36.698612,37.107090,37.512165,39.200000,0.350195
hydration_level,15035.0,78.036991,11.848363,45.000000,69.879145,78.165927,86.498440,100.000000,0.000000
sleep_quality,14927.0,5.817429,1.814602,1.155807,4.510581,5.755650,7.068894,10.000000,0.000000
recovery_score,15420.0,55.202220,17.326338,8.902584,42.321722,54.421403,67.375690,98.000000,0.000000
stress_level,15420.0,0.422916,0.173425,0.100000,0.300472,0.421168,0.543678,0.950000,0.330739
muscle_activity,14850.0,247.724105,126.930007,10.000000,159.150375,245.496682,334.039698,722.542819,0.376135
joint_angles,15420.0,112.464607,30.367148,45.000000,91.437246,112.548535,133.960070,175.000000,0.000000
gait_speed,14989.0,1.864699,0.603927,0.800000,1.430372,1.854269,2.281691,3.500000,0.000000


Low outlier percentiles choosing not to remove any outliers. Using StandardScaler due to low outlier percentages won't pull mean.

In [35]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

In [36]:
X = df.drop(['injury_occurred', 'athlete_id', 'session_id', 'sport_type', 'gender'], axis=1)
y = df['injury_occurred']




X = X.fillna(X.median())

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 10794
Test samples: 4626


In [37]:
num_cols = X_train.select_dtypes(include=["number"]).columns

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, num_cols)],
    remainder="drop",  
)

X_train_ready = preprocessor.fit_transform(X_train)

X_test_ready = preprocessor.transform(X_test)


In [40]:
pd.DataFrame(X_train_ready).to_csv("src/Dataset/X_train.csv", index=False)
pd.DataFrame(X_test_ready).to_csv("src/Dataset/X_test.csv", index=False)
pd.DataFrame(y_train).to_csv("src/Dataset/y_train.csv", index=False)
pd.DataFrame(y_test).to_csv("src/Dataset/y_test.csv", index=False)